# Milvus的基本使用

# 1、DDL操作

## 1.1 数据库相关操作

### ① 查看数据库

举例1：操作客户端

In [1]:

from pymilvus import MilvusClient


client = MilvusClient("http://localhost:19530")

举例2：列出所有数据库

In [2]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default


### ② 创建数据库

In [3]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name=db_name)

### ③ 删除数据库

如果数据库下有Collection则无法删除，需要先删除它的所有Collection才能删除Database

In [7]:

client.drop_database(db_name = db_name)

## 1.2 Collection相关操作

### ① 切换数据库

In [10]:
client.use_database(db_name=db_name)

### ② 查看数据库下的collections

In [29]:
collections = client.list_collections()

for coll in collections:
    print(coll)

docs


### ③ 创建collection

In [15]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ④ 删除collection

In [24]:
client.drop_collection(collection_name=collection_name)

# 2、DML操作

## 2.1 嵌入模型的初始化

## 2.2 准备collection

In [6]:
from common import init_dashscope_embedding_model

embedding_model = init_dashscope_embedding_model()

enable_dynamic_field
### ① 创建collection
- 默认主键名是id
- 默认向量字段是：vector
- 具体结构可以通过schema参数给定，没开动态字段就不能随便添加字段
- 默认开启动态字段,通过enable_dynamic_field开关动态字段

In [7]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE",
    enable_dynamic_field=False
)

### ② 查看collection元数据

In [8]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467911980727184444,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467921963206049812,
    'update_timestamp': 467921963206049812
}

## 2.3 准备数据

### ① 准备原始数据

In [9]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

### ② 生成嵌入向量

In [10]:
vectors = embedding_model.embed_documents(texts)

### ③ 查看生成的嵌入向量

In [11]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
1024
[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575]


### ④ 封装为可以插入的数据格式
- id和vector是必需字段，在没有指定schema情况下，其余字段是动态字段,非必需

In [37]:
data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

## 2.4 写入数据

### ① 修改或插入数据
- 如果id已经存在，则修改，不存在，则新增
- 修改时，旧数据软删除

In [38]:
insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### ② 手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [39]:
client.flush(collection_name=collection_name)

### ③ 查看collection统计信息
- stats中的row_count并不能实时更新，要等后台compaction才会显示正确行数，否则更新时软删除数据也会统计
- query(count(*))能实时显示正确条数（不包括软删除数据）

In [40]:
stats = client.get_collection_stats(collection_name=collection_name)

print("stats : ",stats)

res = client.query(
    collection_name=collection_name,
    output_fields=['count(*)']
)

print(res)

stats :  {'row_count': 8}
data: ["{'count(*)': 4}"], extra_info: {}


# 3、DQL操作

## 3.1 扫描数据
- batch_size可指定每批数量，这里的rows就是每批的数据，每次next()获取一批数据，再遍历next()的结果，获取每条实际数据

In [41]:

iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"],
    batch_size=2
)


while True:
    i = 0

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row.get('text', None)},source = {row.get('source', None)}")

        i += 1

iterator.close()

第1条数据：
id : 0,vector = [0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [0.031253933906555176, -0.0017967253224924207, 0.021053170785307884, 0.0021241146605461836, -0.05789622291922569],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第1条数据：
id : 2,vector = [0.029745042324066162, -0.0020526202861219645, 0.03673800081014633, 0.015409480780363083, -0.017944427207112312],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第2条数据：
id : 3,vector = [0.025592587888240814, -0.007320123258978128, 0.046160709112882614, -0.017560871317982674, -0.02144678682088852],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


## 3.2 通过主键查询数据

In [42]:

res = client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


3
第1条数据：
id : 0,vector = [0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [0.031253933906555176, -0.0017967253224924207, 0.021053170785307884, 0.0021241146605461836, -0.05789622291922569],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [0.029745042324066162, -0.0020526202861219645, 0.03673800081014633, 0.015409480780363083, -0.017944427207112312],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


## 3.3 相似度检索

### ① 准备查询嵌入

In [46]:
# 相似度检索
# query = "什么是向量数据库？"
query = "Milvus是什么"
query_vector = embedding_model.embed_query(query)
print(query_vector[:5])

[0.03295276686549187, -0.0013472350547090173, 0.05690922960639, 0.004592918325215578, -0.030147740617394447]


### ② 检索

In [1]:
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]:
    print(res)

NameError: name 'client' is not defined